# 2 Preprocessing Data

Read in the Google Trends keyword interest data for preprocessing.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
DATA_DIR = Path.cwd()
TRENDS_PATH = DATA_DIR / "trends_final_keywords.csv"

trends = pd.read_csv(TRENDS_PATH, parse_dates=["date"])
trends.head()

,date,keyword,interest
0,2016-05-01,home care,100.0
1,2016-06-01,home care,60.0
2,2016-07-01,home care,90.0
3,2016-08-01,home care,100.0
4,2016-09-01,home care,90.0


In [3]:
# Pivot the table: keywords as rows, dates as columns, interest values as data
trends_pivot = trends.pivot_table(
    index="keyword", 
    columns="date", 
    values="interest", 
    aggfunc="first"
)

print(f"Shape after pivot: {trends_pivot.shape}")
print(f"Keywords (rows): {trends_pivot.shape[0]:,}")
print(f"Monthly observations (columns): {trends_pivot.shape[1]:,}")
print(f"Date range: {trends_pivot.columns.min().date()} to {trends_pivot.columns.max().date()}")

trends_pivot.head()

Shape after pivot: (1446, 121)
Keywords (rows): 1,446
Monthly observations (columns): 121
Date range: 2016-05-01 to 2026-05-01


date,2016-05-01,2016-06-01,2016-07-01,2016-08-01,2016-09-01,2016-10-01,2016-11-01,2016-12-01,2017-01-01,2017-02-01,...,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01
keyword,,,,,,,,,,,,,,,,,,,,,
5g,9.090909,6.666667,10.0,11.111111,10.0,10.0,4.0,8.333333,7.692308,7.692308,...,50.0,45.454545,34.482759,48.275862,56.0,62.962963,82.758621,41.025641,100.000,84.210526
5g network,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,3.448276,0.0,3.846154,3.333333,2.564103,3.125,0.000000
5g spectrum,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000
5g stock,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000
5g technology,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.000000,...,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000,0.000000


In [4]:
# Filter 1: Remove keywords with 70% or more NaN values
nan_share_by_keyword = trends_pivot.isna().sum(axis=1) / len(trends_pivot.columns)
keywords_with_high_nan = nan_share_by_keyword[nan_share_by_keyword >= 0.70].index

print(f"Filter 1: NaN Removal (70%+ threshold)")
print(f"Keywords with 70%+ NaN: {len(keywords_with_high_nan):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_with_high_nan)]
print(f"Keywords remaining: {len(trends_pivot):,}")


Filter 1: NaN Removal (70%+ threshold)
Keywords with 70%+ NaN: 0
Keywords remaining: 1,446


In [5]:
# Filter 2: Remove keywords with 70% or more zero values
zero_share_by_keyword = (trends_pivot == 0).sum(axis=1) / len(trends_pivot.columns)
keywords_with_high_zeros = zero_share_by_keyword[zero_share_by_keyword >= 0.70].index

print(f"\nFilter 2: Zero Removal (70%+ threshold)")
print(f"Keywords with 70%+ zeros: {len(keywords_with_high_zeros):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_with_high_zeros)]
print(f"Keywords remaining: {len(trends_pivot):,}")



Filter 2: Zero Removal (70%+ threshold)
Keywords with 70%+ zeros: 967
Keywords remaining: 479


In [6]:
# Check mean interest statistics AFTER filtering
mean_interest_by_keyword = trends_pivot.mean(axis=1)

print(f"\nMean Interest Statistics (after NaN and zero filtering):")
print(f"Min: {mean_interest_by_keyword.min():.2f}")
print(f"Max: {mean_interest_by_keyword.max():.2f}")
print(f"Mean: {mean_interest_by_keyword.mean():.2f}")
print(f"Median: {mean_interest_by_keyword.median():.2f}")

# Summary table for different thresholds
thresholds = [0.5, 1, 2, 5, 10, 20, 50]
mean_filter_summary = []

for threshold in thresholds:
    keywords_below_threshold = mean_interest_by_keyword[mean_interest_by_keyword < threshold]
    keywords_remaining = len(trends_pivot) - len(keywords_below_threshold)
    
    mean_filter_summary.append({
        "mean_interest_threshold": threshold,
        "keywords_dropped": len(keywords_below_threshold),
        "keywords_remaining": keywords_remaining,
    })

mean_filter_summary_df = pd.DataFrame(mean_filter_summary)
print(f"\nMean Interest Filtering Summary:")
print(mean_filter_summary_df.to_string(index=False))



Mean Interest Statistics (after NaN and zero filtering):
Min: 1.27
Max: 3742.46
Mean: 64.20
Median: 10.29

Mean Interest Filtering Summary:
 mean_interest_threshold  keywords_dropped  keywords_remaining
                     0.5                 0                 479
                     1.0                 0                 479
                     2.0                14                 465
                     5.0                91                 388
                    10.0               234                 245
                    20.0               319                 160
                    50.0               394                  85


In [7]:
# Filter 3 (Optional): Apply mean interest threshold
mean_interest_threshold = 15

keywords_below_threshold = mean_interest_by_keyword[mean_interest_by_keyword < mean_interest_threshold].index

print(f"\nFilter 3: Mean Interest Threshold ({mean_interest_threshold})")
print(f"Keywords with mean interest < {mean_interest_threshold}: {len(keywords_below_threshold):,}")

trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(keywords_below_threshold)]

print(f"Keywords remaining: {len(trends_pivot):,}")
print(f"\nFinal shape: {trends_pivot.shape}")
print(f"Keywords (rows): {trends_pivot.shape[0]:,}")
print(f"Time periods (columns): {trends_pivot.shape[1]:,}")



Filter 3: Mean Interest Threshold (15)
Keywords with mean interest < 15: 287
Keywords remaining: 192

Final shape: (192, 121)
Keywords (rows): 192
Time periods (columns): 121


In [8]:
# Display all keywords for manual review
# Company names should be removed, but "company stock" is OK
print("=" * 80)
print(f"KEYWORDS FOR REVIEW ({len(trends_pivot)} total)")
print("=" * 80)
print("\nRemove standalone company names, but keep '[company] stock' terms\n")

all_keywords = trends_pivot.index.tolist()

for i, keyword in enumerate(all_keywords, 1):
    print(f"{i:3d}. {keyword}")

# Save to CSV for easier review in a spreadsheet
keywords_for_review = pd.DataFrame({
    'keyword': all_keywords,
    'keep': [''] * len(all_keywords),  # Column for manual review
})
keywords_for_review.to_csv(DATA_DIR / 'keywords_for_review.csv', index=False)
print(f"\n\nKeywords saved to 'keywords_for_review.csv' for manual review")


KEYWORDS FOR REVIEW (192 total)

Remove standalone company names, but keep '[company] stock' terms

  1. 5g
  2. advertising
  3. aerospace
  4. airlines
  5. alphabet
  6. aluminum
  7. amazon stock
  8. amd stock
  9. api
 10. apple iphone
 11. apple stock
 12. auto parts
 13. auto sales
 14. automation
 15. bank capital
 16. bank of america
 17. banking
 18. banks
 19. benefits
 20. black friday
 21. broadband
 22. cable tv
 23. cancer treatment
 24. capital gains
 25. carbon fiber
 26. casinos
 27. cement
 28. charity
 29. chemicals
 30. chevron
 31. clay
 32. climate change
 33. coal
 34. comcast
 35. commercial real estate
 36. computing
 37. construction
 38. copper
 39. cost of living
 40. credit card
 41. credit cards
 42. credit score
 43. cryptocurrency
 44. currency
 45. cyber monday
 46. cyber security
 47. defense
 48. depression
 49. derivative
 50. diagnostics
 51. disney plus
 52. disney+
 53. dividend
 54. donation
 55. dow jones
 56. duke energy
 57. earnings
 58. ec

In [9]:
# Manual relevance screen before PCA
# Keep sector/asset/finance terms; remove standalone companies, broad consumer/media terms,
# and duplicate plural variants that should not enter PCA separately.
manual_terms_to_remove = [
    'alphabet',
    'apple iphone',
    'bank of america',
    'chevron',
    'comcast',
    'credit cards',
    'disney plus',
    'disney+',
    'duke energy',
    'exxon',
    'glass',
    'google search',
    'home depot',
    'hotels',
    'meta',
    'movies',
    'netflix',
    'paper',
    'paramount plus',
    'pharmaceuticals',
    'redfin',
    'restaurants',
    'spotify',
    'stock market',
    'tiktok',
    'walmart grocery',
    'wells fargo',
    'youtube',
    'zillow',
]

print("Manual relevance screen:")
print("=" * 60)

keywords_before = len(trends_pivot)
removed_terms = [term for term in manual_terms_to_remove if term in trends_pivot.index]
trends_pivot = trends_pivot.loc[~trends_pivot.index.isin(manual_terms_to_remove)]
keywords_removed = keywords_before - len(trends_pivot)

for term in removed_terms:
    print(f"  ✓ Removed: {term}")

print("=" * 60)
print(f"Manual terms removed: {keywords_removed}")
print(f"Keywords remaining: {len(trends_pivot):,}")
print(f"\nFinal shape: {trends_pivot.shape}")


Manual relevance screen:
  ✓ Removed: alphabet
  ✓ Removed: apple iphone
  ✓ Removed: bank of america
  ✓ Removed: chevron
  ✓ Removed: comcast
  ✓ Removed: credit cards
  ✓ Removed: disney plus
  ✓ Removed: disney+
  ✓ Removed: duke energy
  ✓ Removed: exxon
  ✓ Removed: glass
  ✓ Removed: google search
  ✓ Removed: home depot
  ✓ Removed: hotels
  ✓ Removed: meta
  ✓ Removed: movies
  ✓ Removed: netflix
  ✓ Removed: paper
  ✓ Removed: paramount plus
  ✓ Removed: pharmaceuticals
  ✓ Removed: redfin
  ✓ Removed: restaurants
  ✓ Removed: spotify
  ✓ Removed: stock market
  ✓ Removed: tiktok
  ✓ Removed: walmart grocery
  ✓ Removed: wells fargo
  ✓ Removed: youtube
  ✓ Removed: zillow
Manual terms removed: 29
Keywords remaining: 163

Final shape: (163, 121)


In [10]:
# Save the final filtered dataframe as CSV
output_path = DATA_DIR / 'trends_final_filtered.csv'
trends_pivot.to_csv(output_path)

print(f"Final filtered dataset saved to: trends_final_filtered.csv")
print(f"\nDataset summary:")
print(f"  Shape: {trends_pivot.shape}")
print(f"  Keywords: {trends_pivot.shape[0]:,}")
print(f"  Time periods: {trends_pivot.shape[1]:,}")
print(f"  Date range: {trends_pivot.columns.min().date()} to {trends_pivot.columns.max().date()}")
print(f"\nReady for PCA analysis!")


Final filtered dataset saved to: trends_final_filtered.csv

Dataset summary:
  Shape: (163, 121)
  Keywords: 163
  Time periods: 121
  Date range: 2016-05-01 to 2026-05-01

Ready for PCA analysis!


In [11]:
# Check keywords with highest average interest
mean_interest = trends_pivot.mean(axis=1).sort_values(ascending=False)

print("Top 20 keywords by mean interest:")
print(mean_interest.head(20))

print("\n\nBottom 20 keywords by mean interest:")
print(mean_interest.tail(20))

print(f"\n\nMean interest statistics:")
print(f"Max: {mean_interest.max():.2f}")
print(f"Min: {mean_interest.min():.2f}")
print(f"Mean: {mean_interest.mean():.2f}")
print(f"Median: {mean_interest.median():.2f}")
print(f"Std: {mean_interest.std():.2f}")


Top 20 keywords by mean interest:
keyword
oil             1091.330245
insurance       1034.184814
airlines         802.393502
gold             787.605482
credit card      545.897357
silver           484.717879
benefits         479.570764
real estate      370.824986
wireless         350.378917
steel            320.663138
software         293.470370
construction     257.370553
streaming        246.274471
hardware         236.801272
savings          187.952778
gaming           185.719297
black friday     182.090006
banks            181.912470
dow jones        181.911350
weight loss      174.971594
dtype: float64


Bottom 20 keywords by mean interest:
keyword
supply chain              17.628472
broadband                 17.366083
phosphate                 17.272445
insurance quotes          16.867089
computing                 16.529569
capital gains             16.513958
robotics                  16.487456
silicon valley            16.376469
networking                16.159718
machinery   